# OpenAI Model Inference for MAP Classification

This notebook contains the inference code for various OpenAI models. It uses the OpenAI Python SDK to interact with the models and perform inference on a validation dataset. Finally, it evaluates the performance of each model and compares the results.

<div class="alert-warning">
Libraries
</div>

First, load the nessecary python libraries.

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import os
import time
import tiktoken
from openai import OpenAI
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import getpass


<div class="alert-warning">
Set the working directory and login to OpenAI API
</div>

Second, connect to the OpenAI API, using your personal Open AI API key. 

In [ ]:
# Set working directory 
os.chdir('..\\..\\..\\..\\data')

# If API key is not set in environment variables, prompt the user to enter it
if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = getpass.getpass(prompt='Enter your API key: ')

# Retrieve the API key from environment variables
openai_api_key = os.environ['OPENAI_API_KEY']

# Initialize OpenAI client
client = OpenAI(
  api_key=openai_api_key
)

## Prepare evaluation set

Load the evaluation dataset.

In [ ]:
df = pd.read_excel('GLLM\\evaluation_set_MAP_sentences_final.xlsx')

#Add sentence_id to the dataframe. (This ID is needed to merge the OpenAI outputs back to the inputs)
df['sentence_id'] = f"eval_sentence_" + df.index.astype(str)

## Run MAP inference with different models

Define System and User Prompts.

In [ ]:
### System and User Prompts incl. Implicit, Explicit, Dimension, and Confidence

system_messages = [
  """""",
  """
  You are an AI assistant acting as a Senior Equity Analyst with expertise in Management Accounting Practices (MAP) that analyzes corporate reports for MAP-related methods, tools, and insights.
    
  Please consider the following definition of Management Accounting for your answer:
  "Management accounting is a profession that involves partnering in management decision-making, devising planning and performance management systems,
  and providing expertise in financial reporting and control to assist management in the formulation and implementation of an organization’s strategy."
  """,
  """
  You are an AI assistant acting as a Senior Equity Analyst with expertise in Management Accounting Practices (MAP) that analyzes corporate reports for MAP-related methods, tools, and insights.
    
  Please consider the following definition of Management Accounting for your answer:
  "Management accounting is a profession that involves partnering in management decision-making, devising planning and performance management systems,
  and providing expertise in financial reporting and control to assist management in the formulation and implementation of an organization’s strategy."

  When it comes to different Management Accounting Dimensions consider the following definitions:
  - Budgeting / Planning: Involves preparing and managing budgets, forecasts, and strategic plans to guide resource allocation and align operations with long-term goals. This encompasses business and production planning, budget preparation, and forecasting.
  - Cost: Focuses on measuring, analyzing, and reporting the costs associated with producing goods or services. Activities include cost analysis, application of costing methods, cost allocation, and the development of cost reports and plans to improve efficiency and profitability. 
  - Risk / Internal Control: Covers the identification, assessment, and mitigation of financial and operational risks, as well as the design and assessment of internal control systems. It includes compliance monitoring, risk assessments, and internal audits.
  - Financing / Investment: Relates to decisions about acquiring and allocating financial capital. This includes capital budgeting, investment evaluations, funding strategies, and credit management to support strategic initiatives and long-term growth.
  - Performance / Internal Reporting: Involves measuring and communicating internal performance metrics to support decision-making and continuous improvement. This includes the development of dashboards, KPIs, performance analyses, and internal reports.
  - Strategy: Encompasses activities that support long-term organizational direction, including strategic planning, market positioning, goal setting, and competitive analysis.
  - Operations: Pertains to the planning and monitoring of day-to-day processes that deliver products or services. Management accounting supports operations through inventory management, process improvement, and cost-efficiency initiatives.
  - Pricing & Revenue Management: Involves planning and analyzing pricing strategies and revenue streams. Key areas include product pricing, transfer pricing, revenue forecasting, pricing optimization, and revenue performance analysis.
  """
]
user_messages = [
  """
  You are a Senior Management Accounting Analyst assessing corporate disclosures for Management Accounting Practices (MAP).

  Analyze the <sentence> below and return **only** the following JSON format:
  {
    "Explicit_MAP_referral": "Yes" or "No",
    "Implicit_MAP_referral": "Yes" or "No",
    "Dimension": "One or more values from this list [Budgeting / Planning, Cost, Financing / Investment, Operations, Performance / Internal Reporting, Risk / Internal Control, Strategy, Pricing & Revenue Management] or 'N/A' if both 'Explicit_MAP_referral' and 'Implicit_MAP_referral' are 'No'",
    "Confidence_Score": <an integer between 0 and 100 reflecting confidence that the <sentence> refers to MAPs>
  }
  """,
  """
  You are a Senior Management Accounting Analyst specialized in analyzing corporate disclosure on Management Accounting Practices (MAP).

  Analyze the <sentence> below for relevance to Management Accounting Practices (MAP) using the rules encapsulated in "+++++", then respond **only** in the specified JSON format.
  +++++ [BEGIN OF RULES]

  1. "Explicit_MAP_referral": Output "Yes" if the <sentence> explicitly refers to MAP-related methods, tools or techniques used for internal decision-making, else output "No".

  2. "Implicit_MAP_referral": Output "Yes" if the <sentence> implies the internal use of MAPs for decision-making, even if MAP tools are not explicitly mentioned, else output "No". If "Explicit_MAP_referral" is "Yes", "Implicit_MAP_referral" must also be "Yes".
    
  3. "Dimension": Choose one or more of the following categories (comma-separated in one string) that suits best to the provided <sentence>:
    - "Budgeting / Planning"
    - "Cost"
    - "Financing / Investment"
    - "Operations"
    - "Performance / Internal Reporting"
    - "Risk / Internal Control"
    - "Strategy"
    - "Pricing & Revenue Management"
    **NOTE**: Use "N/A" if both "Explicit_MAP_referral" and "Implicit_MAP_referral" are "No".

  4. "Confidence_Score": Provide your honest confidence score between 0 and 100 representing your confidence that the <sentence> explicitly or implicitly refers to MAPs.

  Return your response **ONLY** in the following JSON format:
  {
    "Explicit_MAP_referral": "Yes" or "No",
    "Implicit_MAP_referral": "Yes" or "No",
    "Dimension": "One or more values from the list of step 3 or 'N/A'",
    "Confidence_Score": <an integer between 0 and 100>
  }
  +++++ [END OF RULES]
  """,
  """
  You are a Senior Management Accounting Analyst specialized in analyzing corporate disclosure on Management Accounting Practices (MAP).

  Analyze the <sentence> below for relevance to Management Accounting Practices (MAP) using the rules encapsulated in "+++++", then respond **only** in the specified JSON format.
  +++++ [BEGIN OF RULES]

  1. "Explicit_MAP_referral": Output "Yes" if the <sentence> explicitly refers to MAP-related methods, tools or techniques used for internal decision-making. 
  Examples include budgeting, cost allocation, forecasting and planning, performance management, internal controls, valuation models (e.g. DCF), hedging programs, or MAP-related systems. 
  Output "No" if the <sentence> focuses solely on external financial reporting, GAAP compliance, legal accruals, or mandatory disclosures without management use.

  2. "Implicit_MAP_referral": Output "Yes" if the <sentence> implies the internal use of MAPs for decision-making, even if MAP tools are not explicitly mentioned. 
  Examples include impairment analysis, sensitivity analysis, performance incentives (e.g. bonus plans), or estimation of (pension) reserves/allowances. 
  Output "No" if the content relates purely to external requirements (e.g., tax, IT security, regulatory laws/regulations/standards) or technical accounting without internal use.
  If "Explicit_MAP_referral" is "Yes", "Implicit_MAP_referral" must also be "Yes".
    
  3. "Dimension": Choose one or more of the following categories (comma-separated in one string) that suits best to the provided <sentence>:
    - "Budgeting / Planning"
    - "Cost"
    - "Financing / Investment"
    - "Operations"
    - "Performance / Internal Reporting"
    - "Risk / Internal Control"
    - "Strategy"
    - "Pricing & Revenue Management"
    **NOTE**: Use "N/A" if both "Explicit_MAP_referral" and "Implicit_MAP_referral" are "No".

  4. "Confidence_Score": Provide your honest confidence score between 0 and 100 representing your confidence that the <sentence> explicitly or implicitly refers to MAPs.
  Use 0 if the sentence is completely unrelated to MAPs, and 100 if MAPs are clearly and explicitly mentioned. Values in between should reflect proportional confidence (e.g., 50 = uncertain or weak implicit relation).

  Return your response **ONLY** in the following JSON format:
  {
    "Explicit_MAP_referral": "Yes" or "No",
    "Implicit_MAP_referral": "Yes" or "No",
    "Dimension": "One or more values from the list of step 3 or 'N/A'",
    "Confidence_Score": <an integer between 0 and 100>
  }
  +++++ [END OF RULES]
  """
]

Define a function to create batch tasks and a function that runs the batch processing of the evaluation set given the batch tasks. In addition, we define a helper function, which can be used to track the job status of the batches.

In [ ]:
# Function to create MAP evaluation tasks for each sentence in the dataset
def create_MAP_batch(data, system_message, user_message, model="gpt-4.1-nano-2025-04-14"):
    tasks = []
    sentence_column = "Sentence"

    for index, row in data.iterrows():
        sentence = row[sentence_column]
        sentence_id = row['sentence_id']  # Access the sentence_id

        if not sentence or pd.isna(sentence):  # Skip empty or NaN sentences
            continue

        if system_message.strip() == "":
            task = {
                "custom_id": f"{sentence_id}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model,
                    "temperature": 0.001,
                    "response_format": {
                        "type": "json_object"
                    },
                    "messages": [
                        {
                            "role": "user",
                            "content": f'{user_message}\n\n Now analyze this <sentence>: {sentence}'
                        }
                    ],
                }
            }
            tasks.append(task)
        
        else:

            task = {
                "custom_id": f"{sentence_id}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model,
                    "temperature": 0.001,
                    "response_format": {
                        "type": "json_object"
                    },
                    "messages": [
                        {
                            "role": "system",
                            "content": system_message
                        },
                        {
                            "role": "user",
                            "content": f'{user_message}\n\n Now analyze this <sentence>: {sentence}'
                        }
                    ],
                }
            }
            tasks.append(task)
    
    return tasks

# Function to process batches: save to .jsonl, upload, and start batch job
def batch_processing(batch, batch_file_name="GLLM\\OpenAI_batch_files\\evaluation_batch.jsonl"):

    batch_file_id = None
    batch_job_id = None

    # 1. Save each batch to a separate .jsonl file

    with open(batch_file_name, 'w') as file:
        for task in batch:
            file.write(json.dumps(task) + '\n')

    
    # 2. Upload batch files to the API
    batch_file = client.files.create(
        file=open(batch_file_name, "rb"),
        purpose="batch"
    )
    print(f"Batch file uploaded: {batch_file.id}")
    batch_file_id = batch_file.id
    
    # 3. Start batch jobs
    batch_job = client.batches.create(
        input_file_id=batch_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )
    print(f"Batch job started: {batch_job.id}")
    batch_job_id = batch_job.id

    return batch_job_id  # Return the batch job ID

# Monitor batch job status and wait for completion (Waiting time between checks: 60 seconds --> can be adjusted: see last line)
def wait_for_batch_to_complete(batch_job_ids):
    while True:
        completed_batches = 0
        failed_batches = 0
        total_batches = len(batch_job_ids)

        # Track failed batch job IDs
        failed_batch_ids = []
        for batch_job_id in batch_job_ids:

            batch_job = client.batches.retrieve(batch_job_id)
            status = batch_job.status
            print(f"Batch Job {batch_job_id} Status: {status}")

            if status == "completed":
                completed_batches += 1
                print(f"Batch Job {batch_job_id} completed successfully.")
                start_time = batch_job.in_progress_at
                end_time = batch_job.completed_at
                if start_time and end_time:
                    duration = (end_time - start_time) / 60 
                    print(f"Batch Job {batch_job_id} duration: {duration} minutes")
                
            if status == "in_progress":
                in_progress_at = batch_job.in_progress_at
                if in_progress_at:
                    readable_time = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(in_progress_at))
                    print(f"Batch Job {batch_job_id} started at: {readable_time}")  
        
            elif status in ["failed", "cancelled", "expired"]:
                failed_batches += 1
                failed_batch_ids.append(batch_job_id)  # Store failed job ID
                print(f"Batch Job {batch_job_id} failed with status: {status}")

        # Print the current status of completed, failed, and outstanding batches
        outstanding_batches = total_batches - (completed_batches + failed_batches)
        print(f"Completed Batches: {completed_batches}, Failed Batches: {failed_batches}, Outstanding Batches: {outstanding_batches}")

        if completed_batches == total_batches:
            print("All batch jobs completed.")
            return
        
        if total_batches == completed_batches + failed_batches:
            print("All batch jobs have either completed or failed.")
            if failed_batch_ids:
                print(f"Failed Batch Job IDs: {failed_batch_ids}")
            return failed_batch_ids

        # Wait before checking again
        print(f"Waiting 60 seconds before checking again...")
        time.sleep(60)


Loop through the different prompt combinations, initilize the batch tasks, and send the batches for processing to the OpenAI server.

In [ ]:
# Create folders for batch files if it doesn't exist
if not os.path.exists("GLLM\\OpenAI_batch_files"):
    os.makedirs("GLLM\\OpenAI_batch_files")
    print("Created folder: GLLM\\OpenAI_batch_files")

#Select the model for evaluation.

#Models without reasoning
# Small: gpt-4.1-nano-2025-04-14
# Medium: gpt-4.1-mini-2025-04-14
# Large: gpt-4.1-2025-04-14

openai_model = "gpt-4.1-2025-04-14"  # Specify the model to use for evaluation

# Define token pricing for cost calculation (in $ per 1 million tokens, retrieved from OpenAI pricing page as of December 2025)
input_token_price_per_1m = {
    "gpt-4.1-nano-2025-04-14": 0.05,
    "gpt-4.1-mini-2025-04-14": 0.20,
    "gpt-4.1-2025-04-14": 1.00
}

output_token_price_per_1m = {
    "gpt-4.1-nano-2025-04-14": 0.20,
    "gpt-4.1-mini-2025-04-14": 0.80,
    "gpt-4.1-2025-04-14": 4.00
}

# Select different prompt combinations to test
prompt_idx = [(1,0), (1,1), (1,2), (2,2), (0,2)] # different combinations of system and user prompts to test
# Explanation of prompt_idx:
# (1,0) = standard system prompt with definition, simple user prompt
# (1,1) = standard system prompt with definition, detailed user prompt with rules
# (1,2) = standard system prompt with definition, most detailed user prompt with rules and examples
# (2,2) = extended system prompt with definition and dimension descriptions, most detailed user prompt with rules and examples
# (0,2) = no system prompt, most detailed user prompt with rules and examples

# Initialize list to store batch job IDs
batch_job_ids = [] 

# Iterate over selected prompt combinations and create/process batches
for system_idx, user_idx in prompt_idx:

    print(f"Uploading file and starting batch for: \n System Prompt {system_idx} and User Prompt {user_idx} \n with model {openai_model}")

    # Select system and user messages based on indices
    system_message = system_messages[system_idx]
    user_message = user_messages[user_idx]

    # Create tasks for the entire evaluation dataset
    Map_tasks = create_MAP_batch(df, system_message, user_message, model=openai_model)

    # Process the batches and get the batch job IDs
    batch_job_id = batch_processing(Map_tasks, batch_file_name=f"GLLM\\OpenAI_batch_files\\evaluation_batch_sys{system_idx}_user{user_idx}_{openai_model}.jsonl")

    # Append the batch job ID to the list
    batch_job_ids.append(batch_job_id)   

Check the status of the batch jobs.

In [ ]:
wait_for_batch_to_complete(batch_job_ids)

Now we define a function to retrieve the batch outputs from the api data server and a function to evaluate the model performance.

In [ ]:
def retrieve_batch_results(batch_job_id, model="gpt-4.1-nano-2025-04-14", system_idx=None, user_idx=None):
    results = []

    # Normalize values
    def normalize(value):
        if isinstance(value, str):
            value = value.strip()
            if value.lower() in ["n/a", "", "na", "none", "nan"]:
                return None
        return value

    # Get batch job status
    batch_job_status = client.batches.retrieve(batch_job_id)
    result_file_id = batch_job_status.output_file_id

    # ! think about error handeling !
    if not result_file_id:
        print(f"No output file found for batch job {batch_job_id}")

    # Download the batch results file
    result_content = client.files.content(result_file_id).content
    result_file_name = f"GLLM\\OpenAI_results\\evaluation_batch_results_sys{system_idx}_user{user_idx}_{model}.jsonl"

    with open(result_file_name, 'wb') as file:
        file.write(result_content)

    print(f"Batch results downloaded: {result_file_name}")

    # Read and parse the results
    with open(result_file_name, 'r') as file:
        for line in file:
            try:
                response = json.loads(line.strip())
                custom_id = response['custom_id']  
                    
                # Extract rating from response
                body = response.get("response", {}).get("body", {})
                choices = body.get("choices", [])
                usage = body.get("usage", {})
                if choices:
                    content = json.loads(choices[0]["message"]["content"])
                    Explicit = normalize(content.get('Explicit_MAP_referral', None))
                    Implicit = normalize(content.get('Implicit_MAP_referral', None))
                    Dimension = normalize(content.get('Dimension', None))
                    Confidence = int(content.get('Confidence_Score', None))

                else:
                    Explicit = None
                    Implicit = None
                    Dimension = None
                    Confidence = None

                if usage:
                    prompt_tokens = usage.get("prompt_tokens", 0)
                    completion_tokens = usage.get("completion_tokens", 0)
                    total_tokens = usage.get("total_tokens", 0)

                # Append results
                results.append({"sentence_id": custom_id, "LLM_Explicit_MAP_referral": Explicit, "LLM_Implicit_MAP_referral": Implicit, "LLM_Dimension": Dimension, "LLM_Confidence_Score": Confidence, "Input_tokens": prompt_tokens, "Output_tokens": completion_tokens, "Total_tokens": total_tokens})

            except Exception as e:
                response = json.loads(line.strip())
                custom_id = response['custom_id']
                print(f"Error parsing result for {custom_id}: {e}")

    return pd.DataFrame(results)

def evaluate_llm(dataset, model_id, system_idx, user_idx,
                          truth_exp_col="Explicit_MAP_referral", pred_exp_col="LLM_Explicit_MAP_referral",
                          truth_imp_col="Implicit_MAP_referral", pred_imp_col="LLM_Implicit_MAP_referral",
                          truth_dimension_col="MAP_dimension_1", pred_dimension_col="LLM_Dimension"):

    # Create a copy of the dataset to avoid modifying the original
    df = dataset.copy()

    # Total Input and Output Tokens
    total_input_tokens = df["Input_tokens"].sum()
    total_output_tokens = df["Output_tokens"].sum()
    total_tokens = df["Total_tokens"].sum()
    input_cost = (total_input_tokens / 1_000_000) * input_token_price_per_1m[model_id]
    output_cost = (total_output_tokens / 1_000_000) * output_token_price_per_1m[model_id]
    total_cost = input_cost + output_cost
    print(f"Total Input Tokens: {total_input_tokens}")
    print(f"Total Output Tokens: {total_output_tokens}")
    print(f"Total Tokens: {total_tokens}")
    print(f"Input Cost: ${input_cost:.4f}")
    print(f"Output Cost: ${output_cost:.4f}")
    print(f"Total Cost: ${total_cost:.4f}")
    
    # Drop rows with missing LLM predictions
    df_exp = df.dropna(subset=[truth_exp_col, pred_exp_col]).copy()
    df_imp = df.dropna(subset=[truth_imp_col, pred_imp_col]).copy()

    print(f"Dropped {len(df)-len(df_exp)} explicit / {len(df)-len(df_imp)} implicit sentences")
    
    #Evaluate Explicit MAP Referral
    print("=== Explicit MAP Referral Evaluation ===")
    if not df_exp.empty:
        exp_accuracy = accuracy_score(df_exp[truth_exp_col], df_exp[pred_exp_col])
        exp_f1_yes = f1_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_precision_yes = precision_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_recall_yes = recall_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="Yes", zero_division=0)
        exp_f1_no = f1_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        exp_precision_no = precision_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        exp_recall_no = recall_score(df_exp[truth_exp_col], df_exp[pred_exp_col], pos_label="No", zero_division=0)
        print("\nClassification Report (Explicit):")
        print(classification_report(df_exp[truth_exp_col], df_exp[pred_exp_col], labels=["Yes", "No"], zero_division=0, digits=3))
    else:
        print("No valid rows for Explicit MAP evaluation.")
    
    #Evaluate Implicit MAP Referral
    print("\n=== Implicit MAP Referral Evaluation ===")
    if not df_imp.empty:
        imp_accuracy = accuracy_score(df_imp[truth_imp_col], df_imp[pred_imp_col])
        imp_precision_yes = precision_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_recall_yes = recall_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_f1_yes = f1_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="Yes", zero_division=0)
        imp_precision_no = precision_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        imp_recall_no = recall_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        imp_f1_no = f1_score(df_imp[truth_imp_col], df_imp[pred_imp_col], pos_label="No", zero_division=0)
        print("\nClassification Report (Implicit):")
        print(classification_report(df_imp[truth_imp_col], df_imp[pred_imp_col], labels=["Yes", "No"], zero_division=0, digits=3))
    else:
        print("No valid rows for Implicit MAP evaluation.")

    #Evaluate MAP Dimension
    print("\n=== MAP Dimension Evaluation ===")
    filtered_dimension = df[
        (df["LLM_Explicit_MAP_referral"] == "No") &
        (df["LLM_Implicit_MAP_referral"] == "No") &
        (~df["LLM_Dimension"].isna())
    ]
    dim_percentage = len(filtered_dimension)/len(df)*100
    print(f"Number of rows where LLM says 'No' to both Explicit and Implicit MAP referral but MAP Dimension is not None: {dim_percentage:.0f}%")

    # Create a list of all dimension columns
    dimension_cols = [col for col in df.columns if col.startswith("MAP_dimension")]
    
    # Check if the micro / macro F1 score for the dimension column 
    label_space = ["Budgeting / Planning", "Cost", "Financing / Investment", "Operations", "Performance / Internal Reporting", "Risk / Internal Control", "Strategy", "Pricing & Revenue Management"]
    
    def encode_labels(text, label_space):
        labels = [l.strip() for l in text.split(",")]
        return [1 if label in labels else 0 for label in label_space]
    
    # join the true cols without empty cells to one column and encode the true and pred dimension cols
    df["dimension_true"] = df[dimension_cols].apply(lambda row: ", ".join(row.dropna().astype(str)), axis=1)
    df["dimension_true_encoded"] = df["dimension_true"].apply(lambda text: encode_labels(text, label_space))
    df["dimension_pred_encoded"] = df[pred_dimension_col].apply(lambda text: encode_labels(text, label_space) if isinstance(text, str) else [0]*len(label_space))

    dimension_micro_f1 = f1_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), average="micro", zero_division=0)
    dimension_macro_f1 = f1_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), average="macro", zero_division=0)
    dimension_accuracy = accuracy_score(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist())
    #full report as table
    dimension_full_report = classification_report(df["dimension_true_encoded"].tolist(), df["dimension_pred_encoded"].tolist(), target_names=label_space, zero_division=0, digits=3)

    # In addition, we check if at least one true dimension is included in the predicted dimensions (which may be a comma-separated string)
    def check_dimension_match(row):
        #true_dim = row[truth_dimension_col]
        true_dims = [row[col] for col in dimension_cols if pd.notnull(row[col])]
        pred_raw = row[pred_dimension_col]
        if all(pd.isnull(true_dims)) and pd.isnull(pred_raw):
            return True  # Both are NaN = match
        elif pd.isnull(pred_raw):
            return False  # No prediction = no match
        else:
            pred_dims = [dim.strip() for dim in pred_raw.split(",")]
            return any(true_dim in pred_dims for true_dim in true_dims)

    if not df.empty:
        print("\nFull per-label Dimension classification report:\n")
        print(dimension_full_report)
        print(f"MAP Dimension Accuracy:{dimension_accuracy:.3f}")
        df["dimension_match"] = df.apply(check_dimension_match, axis=1)
        dimension_accuracy_alternative = df["dimension_match"].mean()
        print(f"MAP Dimension Accuracy (both N/A = match, at least one match): {dimension_accuracy_alternative:.3f}")
    else:
        print("No valid rows for MAP Dimension evaluation.")

    # Return the evaluation results as a dictionary
    return {
        "model_id": model_id,
        "system_idx": system_idx,
        "user_idx": user_idx,
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "dropped_explicit": len(df) - len(df_exp),
        "dropped_implicit": len(df) - len(df_imp),
        "explicit_accuracy": exp_accuracy,
        "explicit_precision_yes": exp_precision_yes,
        "explicit_recall_yes": exp_recall_yes,
        "explicit_f1_yes": exp_f1_yes,
        "explicit_precision_no": exp_precision_no,
        "explicit_recall_no": exp_recall_no,
        "explicit_f1_no": exp_f1_no,
        "implicit_accuracy": imp_accuracy,
        "implicit_precision_yes": imp_precision_yes,
        "implicit_recall_yes": imp_recall_yes,
        "implicit_f1_yes": imp_f1_yes,
        "implicit_precision_no": imp_precision_no,
        "implicit_recall_no": imp_recall_no,
        "implicit_f1_no": imp_f1_no,
        "dimension_percentage_false": dim_percentage,
        "dimension_micro_f1": dimension_micro_f1,
        "dimension_macro_f1": dimension_macro_f1,
        "dimension_accuracy": dimension_accuracy,
        "dimension_accuracy_alternative": dimension_accuracy_alternative,
        "dimension_full_report": dimension_full_report
    }

Retrieve the results from the API database, assess evaluation performance, and save the final results.

In [ ]:
# Create folder for OpenAI results if it doesn't exist
if not os.path.exists("GLLM\\OpenAI_results"):
    os.makedirs("GLLM\\OpenAI_results")
    print("Created folder: GLLM\\OpenAI_results")

# Initialize list to store evaluation results
evaluation_results = []

# Iterate over each batch job ID and corresponding prompt indices
for idx in range(len(prompt_idx)):

    # Get batch job ID and prompt indices
    batch_job_id = batch_job_ids[idx]
    prompt_idx_pair = prompt_idx[idx]
    system_idx = prompt_idx_pair[0]
    user_idx = prompt_idx_pair[1]
    
    # Retrieve results for each batch job
    df_results = retrieve_batch_results(batch_job_id, model=openai_model, system_idx=system_idx, user_idx=user_idx)

    # Merge results with original dataframe
    df_merged = pd.merge(df, df_results, on="sentence_id", how="left")

    # Evaluate LLM performance
    eval_result = evaluate_llm(df_merged, model_id=openai_model, system_idx=system_idx, user_idx=user_idx)
    evaluation_results.append(eval_result)

    # Save the merged dataframe to an Excel file
    df_merged.to_excel(f"GLLM\\OpenAI_results\\output_ZS_sys{system_idx}_user{user_idx}_{openai_model}.xlsx")

# Save evaluation results to a DataFrame and Excel
eval_path = "GLLM\\evaluation_summary_OpenAI_final.xlsx"

# Check if evaluation summary file exists to append results
if os.path.exists(eval_path):
    df_existing = pd.read_excel(eval_path)
    evaluation_results_existing = df_existing.to_dict(orient='records')
    evaluation_results_existing.extend(evaluation_results)
    df_evaluation = pd.DataFrame(evaluation_results_existing)
    df_evaluation.to_excel(f"{eval_path}", index=False)
else:
    print("No existing evaluation summary found. Creating a new one.")
    df_evaluation = pd.DataFrame(evaluation_results)
    df_evaluation.to_excel(f"{eval_path}", index=False)

## Optional: Estimate inference cost of GPT4.1 mini for whole dataset

Last, we estimate the total costs for the OpenAI GLLM interference with zero-shot prompting on the whole dataset using GPT4.1 Mini.

In [ ]:
# Function to count tokens in messages for cost estimation
def count_message_tokens(messages, model="gpt-4.1-nano-2025-04-14"):
    encoding = tiktoken.encoding_for_model(model)
    tokens_per_message = 3  # metadata overhead per message (approx.)
    tokens_per_name = 1     # additional if "name" field present

    total_tokens = 0
    for msg in messages:
        total_tokens += tokens_per_message
        for key, value in msg.items():
            total_tokens += len(encoding.encode(value))
            if key == "name":
                total_tokens += tokens_per_name
    total_tokens += 3  # reply priming
    return total_tokens

In [ ]:
# Load the final cleaned dataframe for batch processing
df_final = pickle.load(open("GLLM\\Corpus_df_HTML_cleaned_GLLM_final.pkl", "rb"))

# create new column for sentence_id
df_final['sentence_ids'] = None

# Create a jsonl file for the whole dataset
file_name = "GLLM\\OpenAI_batch_files\\MAP_sentences_final_ZS.jsonl"

# Select the model for evaluation. 
model = "gpt-4.1-mini-2025-04-14"

# Select the system and user messages for the batch processing
system_message = system_messages[1]
user_message = system_messages[2]

# Open a file to write the processing tasks
processing_file = open(file_name, "w", encoding="utf-8")

for index, row in df_final.iterrows():
    filing_id = f"filing_{index}"

    print(f"Processing filing_id: {filing_id}")

    df_final.at[index, 'filing_id'] = filing_id

    sentence_ids = []

    total_tokens = 0
    for sentence in row['filing_text']:
        
        if not sentence or pd.isna(sentence):  # Skip empty or NaN sentences
            continue

        sentence_id = f"filing_{index}_sentence_{row['filing_text'].index(sentence)}"
        sentence_ids.append(sentence_id)
        task = {
            "custom_id": f"{sentence_id}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": model,
                "temperature": 0.001,
                "response_format": {
                    "type": "json_object"
                },
                "messages": [
                    {
                        "role": "system",
                        "content": system_message
                    },
                    {
                        "role": "user",
                        "content": f'{user_message}\n\n Now analyze this <sentence>: {sentence}'
                    }
                ],
            }
        }
        processing_file.write(json.dumps(task) + "\n")
        processing_file.flush()

        total_tokens += count_message_tokens(task['body']['messages'], model="gpt-4.1-mini-2025-04-14")


    df_final.at[index, 'sentence_ids'] = sentence_ids
    df_final.at[index, 'total_tokens'] = total_tokens

# Close the processing file after writing all tasks
processing_file.close()


In [ ]:
#total input tokens for the whole dataset
total_input_tokens = df_final["total_tokens"].sum()
print(f"{total_input_tokens} total input tokens for the whole dataset. This would take around {total_input_tokens/40_000_000:.2f} days to process at 40M tokens per day (Limit for Tier 3).")

# Cost estimate at 0.2$ per 1M input tokens
cost_estimate = df_final["total_tokens"].sum() / 1_000_000 * 0.2
print(f"Estimated cost at $0.20 per 1M input tokens: ${cost_estimate:.2f}")

#total output tokens at an estimate of 41 tokens per completion
total_prompts = df_final["num_entries"].sum()
total_output_tokens = total_prompts * 41
print(f"{total_output_tokens} total output tokens for the whole dataset")

# Cost estimate at 0.80$ per 1M output tokens
cost_estimate_output = total_output_tokens / 1_000_000 * 0.80
print(f"Estimated cost at $0.80 per 1M output tokens: ${cost_estimate_output:.2f}")

# Total estimated cost
total_estimated_cost = cost_estimate + cost_estimate_output
print(f"Total estimated cost: ${total_estimated_cost:.2f}")